<a href="https://colab.research.google.com/github/81770977/date-fruit-classification-ann/blob/main/Date_Fruit_classification_using_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Date_Fruit_ANN_Clasifier

In [ ]:
import pandas as pd
import numpy as np
df=pd.read_csv("/content/Date_Fruit_Dataset.csv")
df.describe()
df.info()
X=df.drop("Class",axis=1)
y=df["Class"]
df["Class"].unique()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 898 entries, 0 to 897
Data columns (total 35 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   AREA           898 non-null    int64  
 1   PERIMETER      898 non-null    float64
 2   MAJOR_AXIS     898 non-null    float64
 3   MINOR_AXIS     898 non-null    float64
 4   ECCENTRICITY   898 non-null    float64
 5   EQDIASQ        898 non-null    float64
 6   SOLIDITY       898 non-null    float64
 7   CONVEX_AREA    898 non-null    int64  
 8   EXTENT         898 non-null    float64
 9   ASPECT_RATIO   898 non-null    float64
 10  ROUNDNESS      898 non-null    float64
 11  COMPACTNESS    898 non-null    float64
 12  SHAPEFACTOR_1  898 non-null    float64
 13  SHAPEFACTOR_2  898 non-null    float64
 14  SHAPEFACTOR_3  898 non-null    float64
 15  SHAPEFACTOR_4  898 non-null    float64
 16  MeanRR         898 non-null    float64
 17  MeanRG         898 non-null    float64
 18  MeanRB    

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [ ]:
from sklearn.preprocessing import StandardScaler,LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_train
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)
X_test_scaled

array([[-1.46809512, -1.51044055, -1.61098268, ...,  1.71568954,
         1.88635897,  1.38986445],
       [-0.38078401, -0.23043153,  0.05028916, ..., -2.07608033,
        -1.83884423, -1.74766326],
       [-1.37216304, -1.38482587, -1.44927691, ..., -0.09225422,
        -0.11379811, -0.13190525],
       ...,
       [-0.44873279, -0.26550556, -0.1040461 , ...,  0.91537148,
         0.9146128 ,  0.97358961],
       [ 0.58834638,  0.3595581 ,  0.247363  , ...,  0.18830878,
         0.3185332 ,  0.22827726],
       [-1.35551852, -1.22973814, -1.10752165, ...,  1.30474726,
         1.61369423,  1.4415299 ]])

#ANN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset
X_train_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)
X_test_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)
train_dataset=TensorDataset(X_train_tensor,y_train_tensor)
test_dataset=TensorDataset(X_test_tensor,y_test_tensor)
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

#Bulid our model

In [ ]:
class ANN(nn.Module):
  def __init__(self):
    super(ANN,self).__init__()
    self.model=nn.Sequential(
        nn.Linear(X.shape[1],64),
        nn.ReLU(),
        nn.Linear(64,64),
        nn.ReLU(),
        nn.Linear(64,7)
    )
  def forward(self,x):
    return self.model(x)
model=ANN()

#Loss and OPtim


In [ ]:
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

#Training the NN

In [ ]:
epochs=100
for epoch in range(epochs):
  model.train()
  running_loss=0.0
  for xb,yb in train_loader:
    optimizer.zero_grad()
    outputs=model(xb)
    loss=criteria(outputs,yb)
    loss.backward()
    optimizer.step()
    running_loss+=loss
  train_loss=running_loss/len(train_loader)
  print(f"epoch={epoch+1}/{epochs},Loss={train_loss}")

epoch=1/100,Loss=1.6880028247833252
epoch=2/100,Loss=1.0814498662948608
epoch=3/100,Loss=0.7070891261100769
epoch=4/100,Loss=0.5439982414245605
epoch=5/100,Loss=0.46824049949645996
epoch=6/100,Loss=0.42114123702049255
epoch=7/100,Loss=0.3660745620727539
epoch=8/100,Loss=0.33477726578712463
epoch=9/100,Loss=0.29887503385543823
epoch=10/100,Loss=0.27994099259376526
epoch=11/100,Loss=0.25761678814888
epoch=12/100,Loss=0.2419339418411255
epoch=13/100,Loss=0.22844381630420685
epoch=14/100,Loss=0.21425709128379822
epoch=15/100,Loss=0.20615790784358978
epoch=16/100,Loss=0.20362630486488342
epoch=17/100,Loss=0.18955367803573608
epoch=18/100,Loss=0.18184620141983032
epoch=19/100,Loss=0.17307104170322418
epoch=20/100,Loss=0.17216092348098755
epoch=21/100,Loss=0.1702931821346283
epoch=22/100,Loss=0.1610129177570343
epoch=23/100,Loss=0.15259911119937897
epoch=24/100,Loss=0.14685116708278656
epoch=25/100,Loss=0.14178435504436493
epoch=26/100,Loss=0.14566852152347565
epoch=27/100,Loss=0.137799814343

#Evaluate

In [ ]:
model.eval()
total=0
correct=0
with torch.no_grad():
  for xb,yb in test_loader:
    outputs=model(xb)
    _,predicted=torch.max(outputs,1)
    correct+=(predicted==yb).sum().item()
    total+=yb.size(0)
print('Total vals',total)
print("Correct vals",correct)
print("accuracy:",correct/total*100,"%")

Total vals 180
Correct vals 172
accuracy: 95.55555555555556 %
